# COMBINACIÓN ARCHIVOS PROCESADOS DEFUNCIONES

## 0. LIMPIAR MEMORIA

In [109]:
# 0. LIMPIAR MEMORIA ANTES DE INICIAR
import os
import psutil
import gc
def limpiar_memoria():
    """Libera memoria RAM y limpia objetos no usados."""
    gc.collect()
    process = psutil.Process(os.getpid())
    memoria_libre = process.memory_info().rss / (1024 ** 3)
    print(f"🔄 Memoria usada antes de limpieza: {memoria_libre:.2f} GB")

    for var in list(globals().keys()):
        if not var.startswith("_") and var not in ["os", "gc", "psutil", "limpiar_memoria"]:
            del globals()[var]

    gc.collect()
    memoria_final = process.memory_info().rss / (1024 ** 3)
    print(f"✅ Memoria usada después de limpieza: {memoria_final:.2f} GB")
limpiar_memoria()

🔄 Memoria usada antes de limpieza: 2.60 GB
✅ Memoria usada después de limpieza: 2.60 GB


## 1. IMPORTAR LIBRERIAS

In [110]:
import pandas as pd
import os
import gc
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

## 2. DEFINIR FUNCIONES

In [111]:
# Lista global de archivos parquet a combinar
archivos_a_leer = [
    "defunciones_1979_1991_procesado.parquet",
    "defunciones_1992_1996_procesado.parquet",
    "defunciones_1997_1997_procesado.parquet",
    "defunciones_1998_2007_procesado.parquet",
    "defunciones_2008_2011_procesado.parquet",
    "defunciones_2012_2013_procesado.parquet",
    "defunciones_2014_procesado.parquet",
    "defunciones_2015_procesado.parquet",
    "defunciones_2016_procesado.parquet",
    "defunciones_2017_procesado.parquet",
    "defunciones_2018_procesado.parquet",
    "defunciones_2019_procesado.parquet",
    "defunciones_2020_procesado.parquet",
    "defunciones_2021_procesado.parquet",
    "defunciones_2022_procesado.parquet",
    "defunciones_2023_procesado.parquet",
    "defunciones_2024_procesado.parquet"
]

### 2.0. Función para ejecutar combinación alternativa

In [112]:
# ============================================================================
# ENFOQUE ALTERNATIVO SIMPLIFICADO
# ============================================================================

def ejecutar_combinacion_alternativa(lista_dataframes, diccionario, id_estandar):
    """
    Enfoque alternativo más simple y robusto para cuando falla el principal
    """
    print(f"\n{'='*80}")
    print(f"🔄 EJECUTANDO ENFOQUE ALTERNATIVO")
    print(f"{'='*80}")
    
    try:
        # Procesar cada DataFrame individualmente y guardar temporalmente
        archivos_temp = []
        
        for i, (nombre_var, id_archivo) in enumerate(lista_dataframes, 1):
            if nombre_var not in globals():
                print(f"⚠️  [{i}] '{nombre_var}' no encontrado, saltando...")
                continue
                
            print(f"\n🔄 [{i}/{len(lista_dataframes)}] Procesando {nombre_var}...")
            
            df = globals()[nombre_var].copy()
            
            # Aplicar homologación simple
            if id_archivo in diccionario:
                mapeo = diccionario[id_archivo]
                df = df.rename(columns=mapeo)
            
            # Guardar temporalmente
            archivo_temp = f"data/processed/temp_alt_{i:03d}.parquet"
            df.to_parquet(archivo_temp, index=False)
            archivos_temp.append(archivo_temp)
            
            print(f"   ✅ Guardado temporal: {archivo_temp}")
            
            # Limpiar memoria
            del df
            gc.collect()
        
        # Combinar todos los archivos temporales
        print(f"\n🔗 Combinando {len(archivos_temp)} archivos temporales...")
        
        dataframes = []
        for archivo in archivos_temp:
            df_temp = pd.read_parquet(archivo)
            dataframes.append(df_temp)
        
        df_final = pd.concat(dataframes, ignore_index=True)
        
        # Guardar resultado final
        ruta_final = "data/processed/defunciones_completo_alternativo.parquet"
        df_final.to_parquet(ruta_final, index=False)
        
        # Limpiar temporales
        for archivo in archivos_temp:
            try:
                os.remove(archivo)
            except:
                pass
        
        print(f"✅ Enfoque alternativo completado")
        print(f"💾 Archivo guardado en: {ruta_final}")
        
        return df_final
        
    except Exception as e:
        print(f"💥 Error en enfoque alternativo: {e}")
        return None


### 2.1. Función para lectura de archivos

In [113]:
def cargar_archivos_seleccionados_con_id(ruta_carpeta, lista_archivos, engine=None):
    """
    Carga una lista de archivos (Parquet/CSV/Excel) desde ruta_carpeta.
    - Devuelve dict {nombre_archivo: df} (solo los que se pudieron leer)
    """
    import os, pandas as pd
    resultados = {}
    for nombre in lista_archivos:
        ruta = os.path.join(ruta_carpeta, nombre)
        if not os.path.exists(ruta):
            print(f"⚠️ No encontrado: {ruta} — se omite")
            continue
        try:
            if nombre.lower().endswith(('.parquet', '.pq')):
                df = pd.read_parquet(ruta, engine=engine) if engine else pd.read_parquet(ruta)
            elif nombre.lower().endswith(('.csv',)):
                df = pd.read_csv(ruta)
            elif nombre.lower().endswith(('.xls', '.xlsx')):
                df = pd.read_excel(ruta)
            else:
                print(f"⚠️ Formato no soportado para {nombre} — se omite")
                continue
            resultados[nombre] = df
            print(f"✅ Cargado: {nombre}  — shape: {df.shape}")
        except Exception as e:
            print(f"❌ Error cargando {nombre}: {e}")
    return resultados



### 2.2. Función para mostrar primeros 5 registros de cada archivo cargado

In [114]:
# Reemplaza la Celda 4
def mostrar_head_archivos(dict_archivos, mostrar_todas_columnas=False, n=5):
    """
    dict_archivos: dict {nombre: df} como devuelve cargar_archivos_seleccionados_con_id
    mostrar_todas_columnas: si True setea pandas option para mostrar todas las columnas
    """
    import pandas as pd
    if mostrar_todas_columnas:
        pd.set_option('display.max_columns', None)
    for nombre, df in dict_archivos.items():
        try:
            print(f"\n--- {nombre} (shape={df.shape}) ---")
            display(df.head(n))
        except Exception as e:
            print(f"⚠️ No se pudo mostrar head de {nombre}: {e}")
    # Restablecer opción por si acaso
    if mostrar_todas_columnas:
        pd.reset_option('display.max_columns')

### 2.3. Función para cargar diccionario de homologacion

In [115]:
def cargar_diccionario_homologacion(ruta_excel, nombre_hoja="Campos Defunciones"):
    """
    Lee un archivo Excel que contiene la homologación de campos.
    Para cada campo (columna), detecta el ÚLTIMO nombre disponible (no nulo)
    y lo usa como nombre estándar.
    
    VALIDACIÓN: Solo procesa filas con ID válido Y al menos un campo con nombre.
    
    Parámetros:
    -----------
    ruta_excel : str
        Ruta completa al archivo Excel de homologación
    nombre_hoja : str
        Nombre de la hoja a leer (por defecto "Campos Defunciones")
    
    Retorna:
    --------
    dict : Diccionario donde la clave es el ID y el valor es otro diccionario
           con los mapeos de columnas antiguas -> columnas nuevas
    dict : Diccionario con los nombres estándar por columna
    """
    try:
        # Leer Excel
        df_homolog = pd.read_excel(ruta_excel, sheet_name=nombre_hoja)
        
        print(f"\n🔍 DEBUG - Información del Excel:")
        print(f"   Dimensiones originales: {df_homolog.shape}")
        print(f"   Columnas totales: {len(df_homolog.columns)}")
        
        # Verificar que existe columna ID
        if 'ID' not in df_homolog.columns:
            print("❌ ERROR: No se encontró la columna 'ID' en el Excel")
            return None, None
        
        # PASO 1: FILTRAR Y VALIDAR FILAS
        print(f"\n🔍 Validando filas del Excel...")
        
        # Filtrar filas que tengan ID válido (numérico y no nulo)
        df_homolog = df_homolog[df_homolog['ID'].notna()].copy()
        
        # Convertir ID a entero (esto también filtra IDs no numéricos)
        try:
            df_homolog['ID'] = pd.to_numeric(df_homolog['ID'], errors='coerce')
            df_homolog = df_homolog[df_homolog['ID'].notna()].copy()
            df_homolog['ID'] = df_homolog['ID'].astype(int)
        except Exception as e:
            print(f"❌ Error al procesar columna ID: {e}")
            return None, None
        
        # Obtener las columnas de campos (todas excepto 'ID')
        columnas_campos = [col for col in df_homolog.columns if col != 'ID']
        
        # VALIDACIÓN CRÍTICA: Solo mantener filas que tengan al menos UN campo con valor
        print(f"   Filas con ID válido: {len(df_homolog)}")
        
        # Contar valores no nulos en las columnas de campos
        df_homolog['_campos_validos'] = df_homolog[columnas_campos].notna().sum(axis=1)
        
        # Filtrar: solo filas con al menos 1 campo válido
        filas_con_datos = df_homolog[df_homolog['_campos_validos'] > 0].copy()
        filas_sin_datos = df_homolog[df_homolog['_campos_validos'] == 0]
        
        print(f"   ├─ Con datos en campos: {len(filas_con_datos)}")
        print(f"   └─ Sin datos (descartadas): {len(filas_sin_datos)}")
        
        if len(filas_sin_datos) > 0:
            ids_descartados = sorted(filas_sin_datos['ID'].tolist())
            if len(ids_descartados) <= 10:
                print(f"      IDs descartados: {ids_descartados}")
            else:
                print(f"      IDs descartados (primeros 10): {ids_descartados[:10]}")
                print(f"      ... y {len(ids_descartados) - 10} más")
        
        # Usar solo filas válidas
        df_homolog = filas_con_datos.drop('_campos_validos', axis=1)
        
        if len(df_homolog) == 0:
            print("❌ ERROR: No hay filas válidas para procesar")
            return None, None
        
        # Ordenar por ID
        df_homolog = df_homolog.sort_values('ID')
        
        print(f"\n   ✅ Filas válidas a procesar: {len(df_homolog)}")
        print(f"   📋 Rango de IDs: {int(df_homolog['ID'].min())} - {int(df_homolog['ID'].max())}")
        print(f"   📋 Campos a procesar: {len(columnas_campos)}")
        
        # PASO 2: DETECTAR NOMBRES ESTÁNDAR (último valor disponible por columna)
        print(f"\n🔍 Detectando nombres estándar (último valor disponible por columna)...")
        nombres_estandar = {}
        
        for campo in columnas_campos:
            # Obtener todos los valores no nulos de esta columna, en orden de ID
            valores_no_nulos = df_homolog[df_homolog[campo].notna()][campo].tolist()
            
            if valores_no_nulos:
                # El último valor no nulo es el nombre estándar
                ultimo_valor = str(valores_no_nulos[-1]).strip()
                if ultimo_valor:
                    nombres_estandar[campo] = ultimo_valor
                    # Mostrar solo algunos ejemplos
                    if len(nombres_estandar) <= 5:
                        print(f"   ✓ '{campo}' → '{ultimo_valor}'")
        
        if len(nombres_estandar) > 5:
            print(f"   ... y {len(nombres_estandar) - 5} campos más")
        
        print(f"\n   📊 Total nombres estándar detectados: {len(nombres_estandar)}")
        
        # PASO 3: CREAR DICCIONARIO DE HOMOLOGACIÓN POR ID
        print(f"\n🔄 Creando mapeos de homologación por ID...")
        diccionario = {}
        
        ids_procesados = []
        ids_sin_mapeos = []
        
        for idx, row in df_homolog.iterrows():
            id_actual = int(row['ID'])
            mapeo = {}
            
            # Para cada campo, mapear: nombre_actual → nombre_estandar
            for campo in columnas_campos:
                nombre_actual = row[campo]
                
                # Solo procesar si hay un nombre estándar definido para este campo
                if campo in nombres_estandar:
                    nombre_estandar = nombres_estandar[campo]
                    
                    # Si hay un valor actual (no nulo) y es diferente del estándar
                    if pd.notna(nombre_actual):
                        nombre_actual_str = str(nombre_actual).strip()
                        
                        if nombre_actual_str and nombre_actual_str != nombre_estandar:
                            mapeo[nombre_actual_str] = nombre_estandar
            
            diccionario[id_actual] = mapeo
            ids_procesados.append(id_actual)
            
            if len(mapeo) == 0:
                ids_sin_mapeos.append(id_actual)
        
        # REPORTE DE MAPEOS
        print(f"\n   ✅ IDs procesados: {len(ids_procesados)}")
        
        ids_con_mapeos = [id_val for id_val in ids_procesados if id_val not in ids_sin_mapeos]
        print(f"   ├─ Con mapeos (campos a renombrar): {len(ids_con_mapeos)}")
        print(f"   └─ Sin mapeos (nombres ya estándar): {len(ids_sin_mapeos)}")
        
        if ids_sin_mapeos:
            if len(ids_sin_mapeos) <= 10:
                print(f"      IDs sin mapeos: {ids_sin_mapeos}")
            else:
                print(f"      IDs sin mapeos (primeros 10): {ids_sin_mapeos[:10]}")
        
        # Mostrar ejemplos de mapeos
        print(f"\n   📋 Ejemplos de mapeos creados:")
        ejemplos_mostrados = 0
        for id_val in sorted(ids_con_mapeos)[:3]:
            if id_val in diccionario and diccionario[id_val]:
                print(f"      ID {id_val}: {len(diccionario[id_val])} campos a renombrar")
                # Mostrar 2 ejemplos de renombramientos
                ejemplos = list(diccionario[id_val].items())[:2]
                for orig, nuevo in ejemplos:
                    print(f"         '{orig}' → '{nuevo}'")
                ejemplos_mostrados += 1
                if ejemplos_mostrados >= 3:
                    break
        
        print(f"\n{'='*80}")
        print(f"✅ DICCIONARIO DE HOMOLOGACIÓN CARGADO EXITOSAMENTE")
        print(f"{'='*80}")
        print(f"📋 IDs válidos procesados: {len(diccionario)}")
        print(f"📋 Campos estándar definidos: {len(nombres_estandar)}")
        print(f"✅ Listo para homologación")
        print(f"{'='*80}\n")
        
        return diccionario, nombres_estandar
    
    except Exception as e:
        print(f"\n❌ Error al cargar el diccionario de homologación: {e}")
        import traceback
        print(f"📍 Detalles del error:")
        traceback.print_exc()
        return None, None

### 2.4. Función para homologar y combinar dataframes

#### 2.4.1. Homologar V1

In [116]:
def homologar_y_combinar_defunciones_optimizado(lista_dataframes, diccionario_homolog, nombres_estandar, 
                                                 ruta_salida="data/processed/defunciones_completo.parquet",
                                                 batch_size=3):
    """
    Versión optimizada para memoria: procesa y guarda por lotes.
    
    ESTRATEGIA:
    1. Procesa DataFrames en lotes pequeños
    2. Guarda resultados parciales
    3. Combina archivos parciales al final
    4. Reduce uso de RAM significativamente
    
    Parámetros:
    -----------
    batch_size : int
        Número de DataFrames a procesar simultáneamente (default: 3)
        Ajusta según tu RAM disponible
    """
    
    if diccionario_homolog is None:
        print("❌ No hay diccionario de homologación disponible")
        return None
    
    import gc
    
    # ========================================================================
    # FASE 0: ANÁLISIS PREVIO (sin cargar datos completos)
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🚀 PROCESAMIENTO OPTIMIZADO PARA MEMORIA")
    print(f"{'='*80}")
    print(f"📊 Configuración:")
    print(f"   • Total de DataFrames: {len(lista_dataframes)}")
    print(f"   • Tamaño de lote: {batch_size}")
    print(f"   • Lotes necesarios: {(len(lista_dataframes) + batch_size - 1) // batch_size}")
    
    # ========================================================================
    # PASO PREVIO: ANÁLISIS DE COLUMNAS Y MAPEO COMPLETO
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"📊 FASE 1: ANÁLISIS DE COLUMNAS Y CREACIÓN DE MAPEO UNIVERSAL")
    print(f"{'='*80}")
    
    # 1. Crear mapeo universal: nombre_original -> nombre_estándar
    mapeo_universal = {}  # {nombre_original: nombre_final}
    
    # Primero, agregar todos los mapeos del Excel
    for id_archivo, mapeos in diccionario_homolog.items():
        for nombre_original, nombre_estandar in mapeos.items():
            if nombre_original not in mapeo_universal:
                mapeo_universal[nombre_original] = nombre_estandar
            elif mapeo_universal[nombre_original] != nombre_estandar:
                # Conflicto: mismo nombre original mapea a diferentes estándares
                print(f"   ⚠️  CONFLICTO: '{nombre_original}' mapea a:")
                print(f"      • '{mapeo_universal[nombre_original]}' (anterior)")
                print(f"      • '{nombre_estandar}' (ID {id_archivo})")
                print(f"      → Manteniendo: '{mapeo_universal[nombre_original]}'")
    
    # 2. Agregar nombres del Excel que ya son estándar (identidad: nombre -> nombre)
    for nombre_estandar in nombres_estandar:
        if nombre_estandar not in mapeo_universal:
            mapeo_universal[nombre_estandar] = nombre_estandar
    
    # 3. Analizar todas las columnas en todos los DataFrames
    todas_columnas_originales = {}  # {nombre_columna: [lista de IDs donde aparece]}
    
    for nombre_var, id_archivo in lista_dataframes:
        if nombre_var not in globals():
            continue
        
        df_temp = globals()[nombre_var]
        for col in df_temp.columns:
            if col != 'ID':
                if col not in todas_columnas_originales:
                    todas_columnas_originales[col] = []
                todas_columnas_originales[col].append(id_archivo)
    
    # 4. Para columnas NO en el Excel, determinar su nombre final
    # Si una columna aparece en múltiples DataFrames, ese ES su nombre final
    for nombre_col, ids in todas_columnas_originales.items():
        if nombre_col not in mapeo_universal:
            # Esta columna no está en el Excel, usar su nombre tal cual
            mapeo_universal[nombre_col] = nombre_col
    
    # 5. Crear mapeo inverso para detectar duplicados
    # nombre_final -> [lista de nombres_originales que mapean a él]
    mapeo_inverso = {}
    for nombre_orig, nombre_final in mapeo_universal.items():
        if nombre_final not in mapeo_inverso:
            mapeo_inverso[nombre_final] = []
        if nombre_orig not in mapeo_inverso[nombre_final]:
            mapeo_inverso[nombre_final].append(nombre_orig)
    
    # 6. Detectar y reportar columnas con múltiples orígenes
    print(f"\n📋 MAPEO UNIVERSAL CREADO:")
    print(f"   • Nombres originales únicos: {len(mapeo_universal)}")
    print(f"   • Nombres finales únicos: {len(mapeo_inverso)}")
    
    columnas_multiples_origenes = {final: origenes for final, origenes in mapeo_inverso.items() if len(origenes) > 1}
    if columnas_multiples_origenes:
        print(f"\n   ✅ Columnas con múltiples nombres originales (se consolidarán):")
        for i, (nombre_final, nombres_orig) in enumerate(list(columnas_multiples_origenes.items())[:10], 1):
            print(f"      {i}. '{nombre_final}' ← {nombres_orig}")
        if len(columnas_multiples_origenes) > 10:
            print(f"      ... y {len(columnas_multiples_origenes) - 10} más")
    
    # 7. Determinar todas las columnas finales únicas
    todas_columnas_finales = sorted(list(set(mapeo_universal.values())))
    
    print(f"\n   📊 COLUMNAS FINALES: {len(todas_columnas_finales)}")
    
    # Clasificar columnas finales
    columnas_del_excel_final = [col for col in todas_columnas_finales if col in nombres_estandar]
    columnas_adicionales_final = [col for col in todas_columnas_finales if col not in nombres_estandar]
    
    print(f"      ✅ Del Excel: {len(columnas_del_excel_final)}")
    print(f"      ➕ Adicionales: {len(columnas_adicionales_final)}")
    
    # ========================================================================
    # FASE 2: PROCESAMIENTO POR LOTES
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔄 FASE 2: PROCESAMIENTO POR LOTES")
    print(f"{'='*80}")
    
    archivos_parciales = []
    lote_actual = 0
    
    for i in range(0, len(lista_dataframes), batch_size):
        lote_actual += 1
        batch = lista_dataframes[i:i + batch_size]
        
        print(f"\n{'─'*80}")
        print(f"📦 LOTE {lote_actual}/{(len(lista_dataframes) + batch_size - 1) // batch_size}")
        print(f"   DataFrames: {i+1} a {min(i+batch_size, len(lista_dataframes))}")
        print(f"{'─'*80}")
        
        dfs_procesados = []
        
        for idx_global, (nombre_var, id_archivo) in enumerate(batch, start=i+1):
            if nombre_var not in globals():
                print(f"⚠️  [{idx_global}] '{nombre_var}' no encontrado")
                continue
            
            print(f"\n🔄 [{idx_global}/{len(lista_dataframes)}] {nombre_var} (ID: {id_archivo})")
            
            # Cargar DataFrame
            df = globals()[nombre_var].copy()
            print(f"   📏 Dimensiones originales: {df.shape}")
            print(f"   📋 Columnas originales: {len(df.columns)}")
            
            # Eliminar columna ID interna si existe
            if 'ID' in df.columns:
                df.drop('ID', axis=1, inplace=True)
            
            # ════════════════════════════════════════════════════════════════
            # PASO CLAVE: APLICAR MAPEO UNIVERSAL (no solo del ID actual)
            # ════════════════════════════════════════════════════════════════
            print(f"   🔄 Aplicando mapeo universal de nombres...")
            
            columnas_originales = list(df.columns)
            mapeo_aplicado = {}
            
            for col_original in columnas_originales:
                if col_original in mapeo_universal:
                    nombre_final = mapeo_universal[col_original]
                    if col_original != nombre_final:
                        mapeo_aplicado[col_original] = nombre_final
            
            if mapeo_aplicado:
                print(f"   ✅ Renombrando {len(mapeo_aplicado)} columnas:")
                # Mostrar algunos ejemplos
                ejemplos = list(mapeo_aplicado.items())[:5]
                for orig, final in ejemplos:
                    print(f"      • '{orig}' → '{final}'")
                if len(mapeo_aplicado) > 5:
                    print(f"      ... y {len(mapeo_aplicado) - 5} más")
                
                df.rename(columns=mapeo_aplicado, inplace=True)
            else:
                print(f"   ℹ️  No se requieren renombramientos")
            
            # ════════════════════════════════════════════════════════════════
            # VERIFICAR Y CONSOLIDAR COLUMNAS DUPLICADAS
            # ════════════════════════════════════════════════════════════════
            columnas_duplicadas = df.columns[df.columns.duplicated()].unique()
            if len(columnas_duplicadas) > 0:
                print(f"   ⚠️  Detectadas {len(columnas_duplicadas)} columnas duplicadas:")
                for col_dup in columnas_duplicadas:
                    print(f"      • '{col_dup}'")
                
                print(f"   🔧 Consolidando columnas duplicadas...")
                
                # Para cada columna duplicada, consolidar los valores
                for col_dup in columnas_duplicadas:
                    # Obtener todas las columnas con ese nombre
                    cols_indices = [i for i, col in enumerate(df.columns) if col == col_dup]
                    
                    if len(cols_indices) > 1:
                        # Crear nueva columna consolidada usando coalesce (primer valor no nulo)
                        columnas_a_combinar = [df.iloc[:, idx] for idx in cols_indices]
                        
                        # Combinar usando el primer valor no nulo de cada fila
                        import numpy as np
                        df_consolidado = pd.DataFrame(columnas_a_combinar).T
                        columna_consolidada = df_consolidado.apply(
                            lambda row: row.dropna().iloc[0] if row.notna().any() else None, 
                            axis=1
                        )
                        
                        # Eliminar las columnas duplicadas
                        df = df.drop(df.columns[cols_indices], axis=1)
                        
                        # Agregar la columna consolidada
                        df[col_dup] = columna_consolidada
                
                print(f"   ✅ Columnas consolidadas")
            
            print(f"   📋 Columnas después de renombrar: {len(df.columns)}")
            
            # AGREGAR columnas faltantes (todas las que existen en el universo)
            columnas_actuales = set(df.columns)
            columnas_faltantes = set(todas_columnas_finales) - columnas_actuales
            
            if columnas_faltantes:
                print(f"   ➕ Agregando {len(columnas_faltantes)} columnas faltantes")
                for col in columnas_faltantes:
                    if col not in df.columns:  # Verificación adicional
                        df[col] = None
            
            # VERIFICACIÓN FINAL: Asegurar que no hay duplicados antes de reordenar
            if df.columns.duplicated().any():
                print(f"   ⚠️  Aún hay duplicados, eliminando...")
                df = df.loc[:, ~df.columns.duplicated(keep='first')]
            
            # REORDENAR columnas alfabéticamente para consistencia
            # Usar solo columnas que existen en el DataFrame
            columnas_disponibles = [col for col in todas_columnas_finales if col in df.columns]
            df = df[columnas_disponibles]
            
            # OPTIMIZAR TIPOS DE DATOS (crucial para memoria)
            print(f"   🔧 Optimizando tipos de datos...")
            df = optimizar_tipos_dataframe(df)
            
            print(f"   ✅ Procesado completo: {df.shape}")
            
            dfs_procesados.append(df)
        
        # COMBINAR lote actual
        if dfs_procesados:
            print(f"\n🔗 Combinando {len(dfs_procesados)} DataFrames del lote...")
            
            # Verificar que todos los DataFrames tienen las mismas columnas
            print(f"   🔍 Verificando consistencia de columnas...")
            columnas_por_df = [set(df.columns) for df in dfs_procesados]
            
            # Encontrar columnas comunes
            columnas_comunes = set.intersection(*columnas_por_df) if columnas_por_df else set()
            print(f"   ✅ Columnas comunes: {len(columnas_comunes)}")
            
            # Verificar duplicados en cada DataFrame antes de concatenar
            for i, df in enumerate(dfs_procesados):
                if df.columns.duplicated().any():
                    print(f"   ⚠️  DataFrame {i+1} tiene columnas duplicadas")
                    dfs_procesados[i] = df.loc[:, ~df.columns.duplicated(keep='first')]
            
            df_lote = pd.concat(dfs_procesados, ignore_index=True, sort=False)
            
            # Eliminar duplicados de columnas en el resultado
            if df_lote.columns.duplicated().any():
                print(f"   ⚠️  Lote combinado tiene columnas duplicadas, eliminando...")
                df_lote = df_lote.loc[:, ~df_lote.columns.duplicated(keep='first')]
            
            print(f"   ✅ Lote combinado: {df_lote.shape}")
            
            # GUARDAR lote parcial
            archivo_parcial = f"data/processed/temp_lote_{lote_actual:03d}.parquet"
            os.makedirs("data/processed", exist_ok=True)
            
            print(f"   💾 Guardando lote parcial...")
            # Limpiar DataFrame antes de guardar
            df_lote_limpio = limpiar_dataframe_para_parquet(df_lote)
            df_lote_limpio.to_parquet(archivo_parcial, index=False, engine='pyarrow', compression='snappy')
            
            tamaño_mb = os.path.getsize(archivo_parcial) / (1024 * 1024)
            print(f"   ✅ Guardado: {archivo_parcial} ({tamaño_mb:.2f} MB)")
            
            archivos_parciales.append(archivo_parcial)
            
            # LIBERAR MEMORIA
            del df_lote, dfs_procesados
            gc.collect()
            print(f"   🧹 Memoria liberada")
    
    # ========================================================================
    # FASE 3: COMBINACIÓN FINAL DE LOTES
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔗 FASE 3: COMBINACIÓN FINAL")
    print(f"{'='*80}")
    
    print(f"\n📦 Archivos parciales generados: {len(archivos_parciales)}")
    
    if len(archivos_parciales) == 0:
        print("❌ No hay archivos parciales para combinar")
        return None
    
    if len(archivos_parciales) == 1:
        # Si solo hay un lote, renombrar directamente
        print(f"✅ Solo un lote generado, renombrando...")
        os.rename(archivos_parciales[0], ruta_salida)
        df_final = pd.read_parquet(ruta_salida)
    else:
        # Combinar archivos parciales
        print(f"🔄 Combinando {len(archivos_parciales)} archivos parciales...")
        
        # Leer y combinar de forma eficiente
        dfs_parciales = []
        for i, archivo in enumerate(archivos_parciales, 1):
            print(f"   📂 Leyendo lote {i}/{len(archivos_parciales)}...")
            df_temp = pd.read_parquet(archivo)
            dfs_parciales.append(df_temp)
        
        print(f"   🔗 Concatenando todos los lotes...")
        df_final = pd.concat(dfs_parciales, ignore_index=True, sort=False)
        
        # Limpiar
        del dfs_parciales
        gc.collect()
        
        print(f"   💾 Guardando archivo final...")
        # Limpieza de tipos antes de guardar
        for col in df_final.columns:
            # Si una columna tiene mezcla de tipos, la convertimos a string de forma segura
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', None)
            # Si es float pero debería ser código (como C_MUERTE), convertimos también
            elif col == 'C_MUERTE':
                df_final[col] = df_final[col].astype('Int64').astype(str).replace('<NA>', None)

        # Reintento de guardado con manejo de errores
        try:
            print(f"   💾 Guardando archivo final...")
            df_final.to_parquet(ruta_salida, index=False, engine='pyarrow', compression='snappy')
        except Exception as e:
            print(f"❌ Error al guardar Parquet: {e}")
            print("Intentando guardar con conversión completa a string...")
            df_final = df_final.astype(str)
            df_final.to_parquet(ruta_salida, index=False, engine='pyarrow', compression='snappy')
    
    # Limpiar archivos temporales
    print(f"\n🧹 Limpiando archivos temporales...")
    for archivo in archivos_parciales:
        try:
            if os.path.exists(archivo):
                os.remove(archivo)
                print(f"   ✅ Eliminado: {archivo}")
        except Exception as e:
            print(f"   ⚠️  No se pudo eliminar {archivo}: {e}")
    
    # ========================================================================
    # ESTADÍSTICAS FINALES
    # ========================================================================
    tamaño_final_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
    
    print(f"\n{'='*80}")
    print(f"✅ ¡PROCESO COMPLETADO EXITOSAMENTE!")
    print(f"{'='*80}")
    print(f"📊 DATAFRAME FINAL:")
    print(f"   📏 Dimensiones: {df_final.shape}")
    print(f"   📋 Columnas: {len(df_final.columns)}")
    print(f"   📄 Registros: {len(df_final):,}")
    print(f"   💾 Tamaño archivo: {tamaño_final_mb:.2f} MB")
    print(f"   📁 Ubicación: {ruta_salida}")
    
    # Estadísticas de completitud
    print(f"\n📈 COMPLETITUD:")
    total_celdas = df_final.shape[0] * df_final.shape[1]
    total_nulos = df_final.isnull().sum().sum()
    porcentaje_datos = ((total_celdas - total_nulos) / total_celdas) * 100
    
    print(f"   • Celdas totales: {total_celdas:,}")
    print(f"   • Con datos: {porcentaje_datos:.2f}%")
    print(f"   • NULL: {100-porcentaje_datos:.2f}%")
    
    print(f"\n{'='*80}\n")
    
    return df_final

#### 2.4.2. Homologar V2

In [117]:
# ============================================================================
# FUNCIÓN MEJORADA: HOMOLOGAR Y COMBINAR CON GESTIÓN DE MEMORIA AVANZADA
# ============================================================================

def homologar_y_combinar_defunciones_optimizado_v2(lista_dataframes, diccionario_homolog, nombres_estandar, 
                                                   ruta_salida="data/processed/defunciones_completo.parquet",
                                                   batch_size=1):
    """
    Versión ALTAMENTE optimizada para memoria con:
    - Procesamiento por lotes más pequeños
    - Limpieza agresiva de memoria
    - Manejo robusto de tipos de datos
    - Escritura incremental
    """
    
    if diccionario_homolog is None:
        print("❌ No hay diccionario de homologación disponible")
        return None
    
    import gc
    import numpy as np
    
    # ========================================================================
    # CONFIGURACIÓN DE MEMORIA
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🚀 PROCESAMIENTO ULTRA-OPTIMIZADO PARA MEMORIA")
    print(f"{'='*80}")
    print(f"📊 Configuración:")
    print(f"   • Total de DataFrames: {len(lista_dataframes)}")
    print(f"   • Tamaño de lote: {batch_size}")
    print(f"   • Estrategia: Procesamiento incremental + limpieza agresiva")
    
    # ========================================================================
    # FASE 1: ANÁLISIS PREVIO (SIN CARGAR DATOS COMPLETOS)
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"📊 FASE 1: ANÁLISIS DE ESTRUCTURA")
    print(f"{'='*80}")
    
    # Crear mapeo universal
    mapeo_universal = {}
    for id_archivo, mapeos in diccionario_homolog.items():
        for nombre_original, nombre_estandar in mapeos.items():
            if nombre_original not in mapeo_universal:
                mapeo_universal[nombre_original] = nombre_estandar
    
    # Agregar nombres estándar
    for nombre_estandar in nombres_estandar:
        if nombre_estandar not in mapeo_universal:
            mapeo_universal[nombre_estandar] = nombre_estandar
    
    # Analizar columnas existentes
    todas_columnas_originales = set()
    for nombre_var, id_archivo in lista_dataframes:
        if nombre_var in globals():
            df_temp = globals()[nombre_var]
            todas_columnas_originales.update(df_temp.columns)
    
    # Completar mapeo universal
    for col in todas_columnas_originales:
        if col not in mapeo_universal:
            mapeo_universal[col] = col
    
    # Determinar columnas finales
    todas_columnas_finales = sorted(list(set(mapeo_universal.values())))
    
    print(f"   ✅ Mapeo universal: {len(mapeo_universal)} nombres → {len(todas_columnas_finales)} columnas finales")
    
    # ========================================================================
    # FASE 2: PROCESAMIENTO POR LOTES CON GESTIÓN DE MEMORIA
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔄 FASE 2: PROCESAMIENTO POR LOTES CON GESTIÓN DE MEMORIA")
    print(f"{'='*80}")
    
    archivos_parciales = []
    total_procesado = 0
    
    for i in range(0, len(lista_dataframes), batch_size):
        lote_num = (i // batch_size) + 1
        total_lotes = (len(lista_dataframes) + batch_size - 1) // batch_size
        
        print(f"\n{'─'*80}")
        print(f"📦 LOTE {lote_num}/{total_lotes}")
        print(f"   DataFrames: {i+1} a {min(i+batch_size, len(lista_dataframes))}")
        print(f"{'─'*80}")
        
        batch = lista_dataframes[i:i + batch_size]
        dfs_procesados = []
        
        for idx_global, (nombre_var, id_archivo) in enumerate(batch, start=i+1):
            if nombre_var not in globals():
                print(f"⚠️  [{idx_global}] '{nombre_var}' no encontrado")
                continue
            
            print(f"\n🔄 [{idx_global}/{len(lista_dataframes)}] {nombre_var}")
            
            # Cargar y procesar DataFrame
            df = globals()[nombre_var].copy()
            print(f"   📏 Original: {df.shape}")
            
            # LIMPIAR MEMORIA INMEDIATAMENTE
            del globals()[nombre_var]
            gc.collect()
            
            # Eliminar columna ID si existe
            if 'ID' in df.columns:
                df = df.drop('ID', axis=1)
            
            # Aplicar renombramiento
            columnas_renombradas = 0
            for col_original in list(df.columns):
                if col_original in mapeo_universal:
                    nombre_final = mapeo_universal[col_original]
                    if col_original != nombre_final:
                        df = df.rename(columns={col_original: nombre_final})
                        columnas_renombradas += 1
            
            if columnas_renombradas > 0:
                print(f"   🔄 Renombradas {columnas_renombradas} columnas")
            
            # CONSOLIDAR COLUMNAS DUPLICADAS
            if df.columns.duplicated().any():
                print(f"   🔧 Consolidando columnas duplicadas...")
                for col_dup in df.columns[df.columns.duplicated()].unique():
                    cols_indices = [i for i, col in enumerate(df.columns) if col == col_dup]
                    if len(cols_indices) > 1:
                        # Combinar columnas duplicadas
                        columnas_a_combinar = [df.iloc[:, idx] for idx in cols_indices]
                        df_consolidado = pd.DataFrame(columnas_a_combinar).T
                        columna_consolidada = df_consolidado.apply(
                            lambda row: next((x for x in row if pd.notna(x)), None), 
                            axis=1
                        )
                        # Eliminar duplicados y agregar consolidada
                        df = df.drop(df.columns[cols_indices], axis=1)
                        df[col_dup] = columna_consolidada
            
            # AGREGAR COLUMNAS FALTANTES (OPTIMIZADO)
            columnas_faltantes = set(todas_columnas_finales) - set(df.columns)
            if columnas_faltantes:
                print(f"   ➕ Agregando {len(columnas_faltantes)} columnas faltantes")
                # Crear DataFrame temporal con columnas faltantes
                df_faltantes = pd.DataFrame({col: [None] * len(df) for col in columnas_faltantes})
                # Concatenar una sola vez
                df = pd.concat([df, df_faltantes], axis=1)
            
            # REORDENAR COLUMNAS
            columnas_disponibles = [col for col in todas_columnas_finales if col in df.columns]
            df = df[columnas_disponibles]
            
            # OPTIMIZACIÓN CRÍTICA DE MEMORIA
            print(f"   🧹 Optimizando memoria...")
            df = optimizar_dataframe_memoria(df)
            
            print(f"   ✅ Procesado: {df.shape}")
            dfs_procesados.append(df)
            total_procesado += len(df)
            
            # LIMPIEZA AGRESIVA
            del df
            gc.collect()
        
        # COMBINAR Y GUARDAR LOTE
        if dfs_procesados:
            print(f"\n🔗 Combinando {len(dfs_procesados)} DataFrames del lote...")
            
            # Combinar con manejo explícito de memoria
            df_lote = pd.concat(dfs_procesados, ignore_index=True, sort=False)
            
            # Eliminar duplicados de columnas
            if df_lote.columns.duplicated().any():
                df_lote = df_lote.loc[:, ~df_lote.columns.duplicated(keep='first')]
            
            print(f"   📏 Lote combinado: {df_lote.shape}")
            
            # GUARDAR CON VALIDACIÓN ROBUSTA
            archivo_parcial = f"data/processed/temp_lote_{lote_num:03d}.parquet"
            os.makedirs("data/processed", exist_ok=True)
            
            print(f"   💾 Guardando lote parcial...")
            guardar_dataframe_seguro(df_lote, archivo_parcial)
            
            tamaño_mb = os.path.getsize(archivo_parcial) / (1024 * 1024)
            print(f"   ✅ Guardado: {archivo_parcial} ({tamaño_mb:.2f} MB)")
            
            archivos_parciales.append(archivo_parcial)
            
            # LIMPIEZA EXTREMA
            del df_lote, dfs_procesados
            gc.collect()
    
    # ========================================================================
    # FASE 3: COMBINACIÓN FINAL OPTIMIZADA
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔗 FASE 3: COMBINACIÓN FINAL OPTIMIZADA")
    print(f"{'='*80}")
    
    if len(archivos_parciales) == 0:
        print("❌ No hay archivos parciales para combinar")
        return None
    
    if len(archivos_parciales) == 1:
        # Si solo hay un lote, renombrar
        print(f"✅ Solo un lote generado, renombrando...")
        os.rename(archivos_parciales[0], ruta_salida)
        df_final = pd.read_parquet(ruta_salida)
    else:
        # Combinar de forma eficiente en memoria
        print(f"🔄 Combinando {len(archivos_parciales)} archivos parciales...")
        
        # Leer y combinar por partes si es muy grande
        chunks = []
        for i, archivo in enumerate(archivos_parciales, 1):
            print(f"   📂 Leyendo lote {i}/{len(archivos_parciales)}...")
            df_temp = pd.read_parquet(archivo)
            chunks.append(df_temp)
        
        print(f"   🔗 Concatenando todos los lotes...")
        df_final = pd.concat(chunks, ignore_index=True, sort=False)
        
        # Limpiar memoria
        del chunks
        gc.collect()
        
        print(f"   💾 Guardando archivo final...")
        guardar_dataframe_seguro(df_final, ruta_salida)
    
    # LIMPIAR TEMPORALES
    print(f"\n🧹 Limpiando archivos temporales...")
    for archivo in archivos_parciales:
        try:
            if os.path.exists(archivo) and archivo != ruta_salida:
                os.remove(archivo)
                print(f"   ✅ Eliminado: {archivo}")
        except Exception as e:
            print(f"   ⚠️  No se pudo eliminar {archivo}: {e}")
    
    # ESTADÍSTICAS FINALES
    tamaño_final_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
    
    print(f"\n{'='*80}")
    print(f"✅ ¡PROCESO COMPLETADO EXITOSAMENTE!")
    print(f"{'='*80}")
    print(f"📊 RESULTADO FINAL:")
    print(f"   📏 Dimensiones: {df_final.shape}")
    print(f"   📋 Columnas: {len(df_final.columns)}")
    print(f"   📄 Registros: {len(df_final):,}")
    print(f"   💾 Tamaño archivo: {tamaño_final_mb:.2f} MB")
    print(f"{'='*80}\n")
    
    return df_final

#### 2.4.3. Homologar V3

In [118]:
# ============================================================================
# FUNCIÓN DE COMBINACIÓN SIMPLIFICADA PARA MEMORIA
# ============================================================================

def combinacion_simplificada_memoria(lista_dataframes, diccionario, id_estandar, ruta_salida):
    """
    Versión ultra-simplificada que prioriza el uso de memoria
    """
    print(f"🔄 Iniciando combinación simplificada...")
    
    # Determinar todas las columnas finales
    todas_columnas = set()
    for nombre_var, id_archivo in lista_dataframes:
        if nombre_var in globals():
            df_temp = globals()[nombre_var]
            # Aplicar mapeo para determinar columnas finales
            if id_archivo in diccionario:
                mapeo = diccionario[id_archivo]
                for col in df_temp.columns:
                    if col in mapeo:
                        todas_columnas.add(mapeo[col])
                    else:
                        todas_columnas.add(col)
            else:
                todas_columnas.update(df_temp.columns)
    
    todas_columnas = sorted(list(todas_columnas))
    print(f"   📋 Columnas finales identificadas: {len(todas_columnas)}")
    
    # Procesar y combinar en chunks
    chunks = []
    
    for i, (nombre_var, id_archivo) in enumerate(lista_dataframes, 1):
        if nombre_var not in globals():
            continue
            
        print(f"   📦 [{i}/{len(lista_dataframes)}] Procesando {nombre_var}...")
        
        df = globals()[nombre_var].copy()
        
        # Aplicar homologación
        if id_archivo in diccionario:
            df = df.rename(columns=diccionario[id_archivo])
        
        # Asegurar todas las columnas
        for col in todas_columnas:
            if col not in df.columns:
                df[col] = None
        
        # Reordenar columnas
        df = df[todas_columnas]
        
        chunks.append(df)
        
        # Liberar memoria cada 3 dataframes
        if i % 3 == 0:
            gc.collect()
    
    # Elimina elementos vacíos o None antes de concatenar
    chunks = [df for df in chunks if df is not None and not df.empty]

    if not chunks:
        raise ValueError("❌ No se pudo concatenar: todos los dataframes estaban vacíos o None.")

    
    # Combinar todo
    print(f"   🔗 Concatenando {len(chunks)} chunks...")
    df_final = pd.concat(chunks, ignore_index=True)
    
    # Guardar
    print(f"   💾 Guardando resultado final...")
    df_final.to_parquet(ruta_salida, index=False)
    
    return df_final


### 2.5. Función para guardar dataframe en formato parquet

In [119]:
# ============================================================================
# FUNCIÓN 3: GUARDAR DATAFRAME EN FORMATO PARQUET
# ============================================================================

def guardar_dataframe_parquet(df, nombre_archivo, ruta_carpeta="data/processed"):
    """
    Guarda un DataFrame en formato Parquet en la carpeta especificada.
    Incluye validación y limpieza de tipos de datos.
    
    Parámetros:
    -----------
    df : DataFrame
        DataFrame a guardar
    nombre_archivo : str
        Nombre del archivo (sin extensión o con extensión .parquet)
    ruta_carpeta : str
        Ruta de la carpeta donde se guardará (por defecto "data/processed")
    
    Retorna:
    --------
    bool : True si se guardó exitosamente, False en caso contrario
    """
    try:
        print(f"\n{'='*80}")
        print(f"💾 PREPARANDO GUARDADO DE ARCHIVO")
        print(f"{'='*80}")
        
        # Crear copia para no modificar el original
        df_guardar = df.copy()
        
        # ====================================================================
        # PASO 1: DIAGNÓSTICO DE TIPOS DE DATOS
        # ====================================================================
        print(f"\n🔍 PASO 1: Diagnosticando tipos de datos...")
        
        tipos_columnas = df_guardar.dtypes.value_counts()
        print(f"\n   Distribución de tipos:")
        for tipo, cantidad in tipos_columnas.items():
            print(f"   • {tipo}: {cantidad} columnas")
        
        # Identificar columnas problemáticas (tipo object)
        columnas_object = df_guardar.select_dtypes(include=['object']).columns.tolist()
        if columnas_object:
            print(f"\n   ⚠️  {len(columnas_object)} columnas tipo 'object' detectadas")
            print(f"      (pueden causar problemas al guardar)")
        
        # ====================================================================
        # PASO 2: LIMPIEZA Y CONVERSIÓN DE TIPOS
        # ====================================================================
        print(f"\n🔧 PASO 2: Limpiando y convirtiendo tipos de datos...")
        
        # Lista de columnas que DEBEN ser numéricas
        columnas_numericas = [
            'ANO', 'MES', 'HORA', 'MINUTOS', 'COD_DPTO', 'COD_MUNIC',
            'CODPTORE', 'CODMUNRE', 'COD_INST', 'EDAD_MADRE',
            'N_HIJOSV', 'N_HIJOSM', 'PESO_NAC', 'T_GES',
            'CODPRES', 'CODPAISNACFAL', 'CODPAISNACMAD', 'CODOCUR', 'CODMUNOC'
        ]
        
        conversiones_exitosas = 0
        conversiones_fallidas = []
        
        for col in columnas_numericas:
            if col in df_guardar.columns:
                try:
                    tipo_actual = df_guardar[col].dtype
                    
                    # Si ya es numérico, continuar
                    if pd.api.types.is_numeric_dtype(df_guardar[col]):
                        conversiones_exitosas += 1
                        continue
                    
                    # Intentar conversión a numérico
                    df_guardar[col] = pd.to_numeric(df_guardar[col], errors='coerce')
                    conversiones_exitosas += 1
                    
                except Exception as e:
                    conversiones_fallidas.append((col, str(e)))
        
        if conversiones_exitosas > 0:
            print(f"   ✅ {conversiones_exitosas} columnas numéricas procesadas")
        
        if conversiones_fallidas:
            print(f"   ⚠️  {len(conversiones_fallidas)} conversiones fallidas:")
            for col, error in conversiones_fallidas[:5]:
                print(f"      • {col}: {error}")
        
        # Convertir columnas object a string
        columnas_convertidas = 0
        for col in columnas_object:
            try:
                if col in df_guardar.columns:
                    # Convertir a string
                    df_guardar[col] = df_guardar[col].astype(str)
                    # Limpiar valores 'nan' string
                    df_guardar[col] = df_guardar[col].replace(['nan', 'None', '<NA>'], None)
                    columnas_convertidas += 1
            except Exception as e:
                print(f"   ⚠️  Error convirtiendo '{col}': {e}")
        
        if columnas_convertidas > 0:
            print(f"   ✅ {columnas_convertidas} columnas text/string procesadas")
        
        # ====================================================================
        # PASO 3: VALIDACIÓN FINAL
        # ====================================================================
        print(f"\n✓ PASO 3: Validación final...")
        
        # Verificar columnas con tipos mixtos
        columnas_problematicas = []
        for col in df_guardar.columns:
            if df_guardar[col].dtype == 'object':
                # Verificar si hay tipos mixtos en la columna
                tipos_unicos = df_guardar[col].dropna().apply(type).unique()
                if len(tipos_unicos) > 1:
                    columnas_problematicas.append((col, tipos_unicos))
        
        if columnas_problematicas:
            print(f"   ⚠️  {len(columnas_problematicas)} columnas con tipos mixtos detectadas")
            print(f"      Convirtiendo todas a string...")
            
            for col, tipos in columnas_problematicas[:10]:
                try:
                    df_guardar[col] = df_guardar[col].astype(str).replace('nan', None)
                except:
                    pass
        
        print(f"   ✅ Validación completada")
        
        # ====================================================================
        # PASO 4: GUARDAR ARCHIVO
        # ====================================================================
        print(f"\n💾 PASO 4: Guardando archivo...")
        
        # Crear la carpeta si no existe
        os.makedirs(ruta_carpeta, exist_ok=True)
        
        # Asegurar extensión .parquet
        if not nombre_archivo.endswith('.parquet'):
            nombre_archivo += '.parquet'
        
        ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
        
        # Intentar guardar
        df_guardar.to_parquet(
            ruta_completa,
            index=False,
            engine='pyarrow',
            compression='snappy'
        )
        
        # Obtener tamaño del archivo
        tamaño_mb = os.path.getsize(ruta_completa) / (1024 * 1024)
        
        print(f"\n{'='*80}")
        print(f"✅ ARCHIVO GUARDADO EXITOSAMENTE")
        print(f"{'='*80}")
        print(f"📁 Ubicación: {ruta_completa}")
        print(f"💾 Tamaño: {tamaño_mb:.2f} MB")
        print(f"📊 Registros: {len(df_guardar):,}")
        print(f"📋 Columnas: {len(df_guardar.columns)}")
        
        # Resumen de tipos finales
        print(f"\n📊 Tipos de datos finales:")
        tipos_finales = df_guardar.dtypes.value_counts()
        for tipo, cantidad in tipos_finales.items():
            print(f"   • {tipo}: {cantidad} columnas")
        
        print(f"{'='*80}\n")
        
        return True
    
    except Exception as e:
        print(f"\n{'='*80}")
        print(f"❌ ERROR AL GUARDAR EL ARCHIVO")
        print(f"{'='*80}")
        print(f"Error: {e}")
        
        # Diagnóstico adicional
        print(f"\n🔍 Diagnóstico del error:")
        
        # Identificar la columna problemática si es posible
        if "Conversion failed for column" in str(e):
            columna_error = str(e).split("column ")[1].split(" ")[0]
            print(f"   • Columna problemática: {columna_error}")
            
            if columna_error in df_guardar.columns:
                print(f"   • Tipo actual: {df_guardar[columna_error].dtype}")
                print(f"   • Valores únicos (primeros 10):")
                valores_unicos = df_guardar[columna_error].dropna().unique()[:10]
                for val in valores_unicos:
                    print(f"      - {val} (tipo: {type(val).__name__})")
                
                # Intentar solución automática
                print(f"\n   🔧 Intentando conversión forzada a string...")
                try:
                    df_temp = df_guardar.copy()
                    df_temp[columna_error] = df_temp[columna_error].astype(str).replace('nan', None)
                    
                    ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
                    df_temp.to_parquet(ruta_completa, index=False, engine='pyarrow', compression='snappy')
                    
                    print(f"   ✅ Guardado exitoso después de conversión")
                    return True
                except Exception as e2:
                    print(f"   ❌ Conversión fallida: {e2}")
        
        print(f"\n💡 Sugerencias:")
        print(f"   1. Verificar tipos de datos en columnas object")
        print(f"   2. Usar df.info() para inspeccionar el DataFrame")
        print(f"   3. Convertir manualmente columnas problemáticas a string")
        print(f"{'='*80}\n")
        
        return False

### 2.6. Función para uso completo

In [120]:
# ============================================================================
# EJEMPLO DE USO COMPLETO
# ============================================================================

def ejemplo_uso_completo():
    """
    Ejemplo completo del flujo de trabajo para homologar y combinar defunciones.
    """
    
    print("="*80)
    print("PROCESO DE HOMOLOGACIÓN Y COMBINACIÓN DE DEFUNCIONES")
    print("="*80)
    
    # PASO 1: Cargar diccionario de homologación
    print("\n📖 PASO 1: Cargando diccionario de homologación...")
    ruta_excel = "ruta/al/archivo/homologacion.xlsx"  # AJUSTAR RUTA
    diccionario, id_estandar = cargar_diccionario_homologacion(ruta_excel, "Campos Defunciones")
    
    if diccionario is None:
        print("❌ No se pudo continuar sin el diccionario")
        return
    
    # PASO 2: Definir lista de DataFrames a procesar
    print("\n🗂️ PASO 2: Definiendo DataFrames a procesar...")
    
    # Lista de tuplas: (nombre_variable, id_correspondiente)
    lista_dataframes = [
        ('defunciones_1979_1991_procesado', 1),
        ('defunciones_1992_1996_procesado', 2),
        ('defunciones_1997_1997_procesado', 3),
        ('defunciones_1998_2007_procesado', 4),
        ('defunciones_2008_2011_procesado', 5),
        ('defunciones_2012_2013_procesado', 6),
        ('defunciones_2014_procesado', 7),
        ('defunciones_2015_procesado', 8),
        ('defunciones_2016_procesado', 9),
        ('defunciones_2017_procesado', 10),
        ('defunciones_2018_procesado', 10),  # Mismo ID que 2017
        ('defunciones_2019_procesado', 12),
        ('defunciones_2020_procesado', 13),
        ('defunciones_2021_procesado', 14),
        ('defunciones_2022_procesado', 15),
        ('defunciones_2023_procesado', 16),
        ('defunciones_2024_procesado', 17)
    ]
    
    # PASO 3: Homologar y combinar
    print("\n🔄 PASO 3: Homologando y combinando DataFrames...")
    defunciones = homologar_y_combinar_defunciones(lista_dataframes, diccionario, id_estandar)
    
    if defunciones is None:
        print("❌ No se pudo crear el DataFrame combinado")
        return
    
    # PASO 4: Guardar resultado
    print("\n💾 PASO 4: Guardando DataFrame combinado...")
    exito = guardar_dataframe_parquet(defunciones, "defunciones_completo", "data/processed")
    
    if exito:
        print("\n" + "="*80)
        print("✅ PROCESO COMPLETADO EXITOSAMENTE")
        print("="*80)
        
        # Mostrar muestra del DataFrame
        print("\n📋 Muestra del DataFrame combinado (primeras 5 filas):")
        print(defunciones.head())
    
    return defunciones


### 2.6. Función convertir lista de archivos a tuplas

In [121]:
# ============================================================================
# FUNCIÓN AUXILIAR: CONVERTIR LISTA DE ARCHIVOS A FORMATO REQUERIDO
# ============================================================================

def convertir_lista_archivos_a_tuplas(lista_archivos):
    """
    Convierte una lista de nombres de archivos .parquet a una lista de tuplas
    con formato (nombre_variable, id_incremental) para usar en homologación.
    
    CONSIDERACIÓN ESPECIAL: Los archivos de 2017 y 2018 comparten el ID 10
    
    Parámetros:
    -----------
    lista_archivos : list
        Lista de nombres de archivos .parquet
        Ejemplo: ["defunciones_2017_procesado.parquet", "defunciones_2018_procesado.parquet"]
    
    Retorna:
    --------
    list of tuples : Lista de tuplas (nombre_variable, id)
    
    Ejemplo de uso:
    ---------------
    archivos_a_leer = [
        "defunciones_2017_procesado.parquet",
        "defunciones_2018_procesado.parquet"
    ]
    lista_tuplas = convertir_lista_archivos_a_tuplas(archivos_a_leer)
    """
    lista_tuplas = []
    
    for idx, archivo in enumerate(lista_archivos, start=1):
        # Remover la extensión .parquet
        nombre_variable = os.path.splitext(archivo)[0]
        
        # ID especial para 2017 y 2018: ambos usan ID 10
        if '2017' in nombre_variable or '2018' in nombre_variable:
            id_asignado = 10
        else:
            id_asignado = idx
        
        # Crear tupla (nombre_variable, id)
        lista_tuplas.append((nombre_variable, id_asignado))
    
    print(f"✅ Conversión completada:")
    print(f"   📁 Archivos procesados: {len(lista_tuplas)}")
    print(f"   🆔 IDs asignados: 1 a {len(lista_tuplas)}")
    print(f"   ⚠️  NOTA: 2017 y 2018 comparten ID 10")
    
    # Mostrar los primeros 3 y últimos 3 para verificación
    print(f"\n📋 Primeros 3 elementos:")
    for i in range(min(3, len(lista_tuplas))):
        print(f"   {lista_tuplas[i]}")
    
    if len(lista_tuplas) > 6:
        print(f"   ...")
        print(f"📋 Últimos 3 elementos:")
        for i in range(max(0, len(lista_tuplas) - 3), len(lista_tuplas)):
            print(f"   {lista_tuplas[i]}")
    
    return lista_tuplas

### 2.7. Función para optimizar tipos de dataframe

In [122]:
def optimizar_tipos_dataframe(df):
    """
    Optimiza tipos de datos para reducir uso de memoria.
    CORREGIDO: Maneja correctamente valores None/NaN para evitar errores de PyArrow.
    """
    # Columnas numéricas que deberían ser enteros
    columnas_int = [
        'ANO', 'MES', 'HORA', 'MINUTOS', 'COD_DPTO', 'COD_MUNIC',
        'CODPTORE', 'CODMUNRE', 'EDAD_MADRE', 'N_HIJOSV', 'N_HIJOSM'
    ]
    
    # Columnas numéricas que pueden ser float
    columnas_float = ['PESO_NAC', 'T_GES']
    
    for col in columnas_int:
        if col in df.columns:
            try:
                # Convertir a numérico
                df[col] = pd.to_numeric(df[col], errors='coerce')
                # Usar Int32 (permite NaN y ahorra memoria)
                if df[col].notna().any():
                    df[col] = df[col].astype('Int32')
            except:
                pass
    
    for col in columnas_float:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df[col] = df[col].astype('float32')
            except:
                pass
    
    # =========================================================================
    # CRÍTICO: Optimizar columnas de texto SIN crear strings 'None'
    # =========================================================================
    columnas_object = df.select_dtypes(include=['object']).columns
    for col in columnas_object:
        try:
            # PASO 1: Reemplazar valores None/NaN con pd.NA ANTES de convertir a string
            df[col] = df[col].fillna(pd.NA)
            
            # PASO 2: Convertir solo valores no-NA a string
            mask_not_na = df[col].notna()
            if mask_not_na.any():
                df.loc[mask_not_na, col] = df.loc[mask_not_na, col].astype(str)
            
            # PASO 3: Limpiar strings problemáticos y volver a None
            df[col] = df[col].replace(['nan', 'None', '<NA>', 'NaN', 'null'], None)
            
            # PASO 4: Si tiene pocas categorías únicas, usar category
            n_unique = df[col].nunique()
            if n_unique < len(df) * 0.5 and n_unique > 0:
                df[col] = df[col].astype('category')
        except Exception as e:
            # Si falla, intentar conversión simple
            try:
                df[col] = df[col].astype(str)
                df[col] = df[col].replace(['nan', 'None', '<NA>'], None)
            except:
                pass
    
    return df

### 2.8. Función para limpiar memoria ram

In [123]:
# Reemplaza el contenido de la Celda 13 por esto
def limpiar_memoria(vars_a_eliminar=None, verbose=True):
    """
    Limpieza segura de memoria:
    - vars_a_eliminar: lista de nombres de variables globales a eliminar (opcional).
    - No borra módulos ni funciones del entorno.
    - Hace gc.collect() y muestra memoria antes/después.
    """
    import gc, os, psutil
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / (1024 ** 3)
    if verbose:
        print(f"🔄 Memoria usada antes de limpieza: {mem_before:.2f} GB")

    # Eliminar solo variables especificadas (si se dio la lista)
    if vars_a_eliminar:
        for v in vars_a_eliminar:
            if v in globals():
                try:
                    del globals()[v]
                except Exception as e:
                    if verbose:
                        print(f"⚠️ No se pudo borrar {v}: {e}")

    # Cerrar/limpiar objetos típicos si existen (DataFrames, conexiones)
    for name in list(globals().keys()):
        if name.startswith("_"):
            continue
        # No eliminar módulos, funciones o nombres del sistema
        if name in ("pd", "np", "os", "gc", "psutil", "limpiar_memoria"):
            continue

    gc.collect()
    mem_after = process.memory_info().rss / (1024 ** 3)
    if verbose:
        print(f"✅ Memoria usada después de limpieza: {mem_after:.2f} GB")

### 2.9. Función para diagnosticar duplicados de columnas

In [124]:
def diagnosticar_duplicados_columnas(lista_dataframes, diccionario_homolog):
    """
    Función de diagnóstico: identifica qué DataFrames tienen columnas duplicadas
    después de aplicar el diccionario de homologación.
    """
    print(f"\n{'='*80}")
    print(f"🔍 DIAGNÓSTICO DE COLUMNAS DUPLICADAS")
    print(f"{'='*80}\n")
    
    problemas_encontrados = []
    
    for idx, (nombre_var, id_archivo) in enumerate(lista_dataframes, 1):
        if nombre_var not in globals():
            continue
        
        df = globals()[nombre_var].copy()
        
        # Aplicar renombramiento
        if id_archivo in diccionario_homolog:
            mapeo = diccionario_homolog[id_archivo]
            for col_antigua, col_nueva in mapeo.items():
                if col_antigua in df.columns:
                    df.rename(columns={col_antigua: col_nueva}, inplace=True)
        
        # Verificar duplicados
        duplicados = df.columns[df.columns.duplicated()].unique()
        
        if len(duplicados) > 0:
            print(f"⚠️  [{idx}] {nombre_var} (ID: {id_archivo})")
            print(f"   Columnas duplicadas: {list(duplicados)}")
            
            # Mostrar todas las apariciones
            for col_dup in duplicados:
                indices = [i for i, col in enumerate(df.columns) if col == col_dup]
                print(f"   '{col_dup}' aparece en posiciones: {indices}")
            
            problemas_encontrados.append((nombre_var, id_archivo, list(duplicados)))
            print()
    
    if not problemas_encontrados:
        print("✅ No se encontraron columnas duplicadas en ningún DataFrame")
    else:
        print(f"\n{'='*80}")
        print(f"📊 RESUMEN: {len(problemas_encontrados)} DataFrames con duplicados")
        print(f"{'='*80}")
        
        for nombre, id_df, dups in problemas_encontrados:
            print(f"• {nombre} (ID {id_df}): {len(dups)} duplicados")
    
    return problemas_encontrados

### 2.10. Función para limpiar dataframe para parquet

In [125]:
def limpiar_dataframe_para_parquet(df):
    """
    Limpia un DataFrame para asegurar compatibilidad con PyArrow/Parquet.
    """
    df_limpio = df.copy()
    
    for col in df_limpio.columns:
        dtype = df_limpio[col].dtype
        
        # Para columnas object, asegurar que no hay strings 'None'
        if dtype == 'object' or str(dtype) == 'string':
            # Reemplazar strings problemáticos con None real
            df_limpio[col] = df_limpio[col].replace(
                ['None', 'nan', 'NaN', '<NA>', 'null', 'NULL'], 
                None
            )
            
            # Si todos son None o vacíos, convertir a string explícitamente
            if df_limpio[col].notna().sum() == 0:
                df_limpio[col] = df_limpio[col].astype(str)
            else:
                # Intentar convertir a string solo valores no-None
                mask = df_limpio[col].notna()
                if mask.any():
                    try:
                        df_limpio.loc[mask, col] = df_limpio.loc[mask, col].astype(str)
                    except:
                        df_limpio[col] = df_limpio[col].astype(str)
                        df_limpio[col] = df_limpio[col].replace('None', None)
    
    return df_limpio

### 2.11. Función para optimizar dataframe para memoria

In [126]:
# ============================================================================
# FUNCIÓN AUXILIAR: OPTIMIZAR DATAFRAME PARA MEMORIA
# ============================================================================

def optimizar_dataframe_memoria(df):
    """
    Optimiza un DataFrame para reducir uso de memoria.
    Maneja específicamente el error de tipos mixtos.
    """
    df_opt = df.copy()
    
    for col in df_opt.columns:
        # Para columnas object, manejar tipos mixtos
        if df_opt[col].dtype == 'object':
            try:
                # Intentar detectar si son numéricas
                converted = pd.to_numeric(df_opt[col], errors='coerce')
                if converted.notna().sum() > len(df_opt) * 0.8:  # Si >80% son numéricos
                    df_opt[col] = converted
                else:
                    # Forzar a string y limpiar
                    df_opt[col] = df_opt[col].astype(str)
                    # Reemplazar strings que representan NaN
                    df_opt[col] = df_opt[col].replace(['nan', 'None', '<NA>', 'NaN', 'null', 'NULL', ''], None)
                    # Si todos son None, usar tipo más eficiente
                    if df_opt[col].isna().all():
                        df_opt[col] = None
            except:
                # Si falla, forzar a string limpio
                df_opt[col] = df_opt[col].astype(str).replace(['nan', 'None', '<NA>'], None)
        
        # Optimizar numéricas
        elif pd.api.types.is_numeric_dtype(df_opt[col]):
            if df_opt[col].notna().all():
                # Si no hay nulos, usar tipo nativo
                if df_opt[col].dtype == 'float64':
                    df_opt[col] = df_opt[col].astype('float32')
                elif df_opt[col].dtype == 'int64':
                    df_opt[col] = df_opt[col].astype('int32')
    
    return df_opt


### 2.12. Función auxiliar para guardar dataframe seguro

In [127]:
# Agregar/pegar esto en la Celda 7 (o en una celda utilitaria)
def guardar_dataframe_seguro(df, ruta_salida, engine_preferido="pyarrow", crear_carpeta=True):
    """
    Guarda DataFrame a parquet de forma robusta:
    - limpia tipos conflictivos
    - intenta pyarrow primero, si falla prueba fastparquet
    - crea carpeta si no existe
    """
    import os, pandas as pd
    if crear_carpeta:
        os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

    # Limpiar tipos problemáticos con la función anterior
    try:
        df_lim = limpiar_tipos(df)
    except Exception as e:
        print(f"⚠️ limpiar_tipos falló: {e} — procediendo con copia básica")
        df_lim = df.copy()

    # Forzar que los object con None sean None en lugar de 'None'
    for col in df_lim.columns[df_lim.dtypes == object]:
        df_lim[col] = df_lim[col].where(df_lim[col].notna(), None)

    # Intentar guardar con pyarrow, si falla intentar fastparquet
    try:
        df_lim.to_parquet(ruta_salida, engine=engine_preferido, index=False)
    except Exception as e1:
        print(f"⚠️ Guardado con {engine_preferido} falló: {e1}")
        fallback = "fastparquet" if engine_preferido == "pyarrow" else "pyarrow"
        try:
            df_lim.to_parquet(ruta_salida, engine=fallback, index=False)
            print(f"✅ Guardado con engine fallback: {fallback}")
        except Exception as e2:
            print(f"❌ Ambos métodos fallaron. Último error: {e2}")
            raise e2
    else:
        print(f"✅ Guardado correcto en {ruta_salida} con {engine_preferido}")


### 2.13. Función para limpiar memoria completa

In [128]:
# ============================================================================
# FUNCIÓN MEJORADA: LIMPIAR MEMORIA
# ============================================================================

def limpiar_memoria_completa():
    """
    Limpieza agresiva de memoria.
    """
    import gc
    import psutil
    import os
    
    process = psutil.Process(os.getpid())
    memoria_inicial = process.memory_info().rss / (1024 ** 3)
    
    # Colectar basura
    gc.collect()
    
    # Limpiar variables globales grandes
    for var_name in list(globals().keys()):
        if not var_name.startswith("_") and var_name not in ['os', 'gc', 'psutil', 'pd', 'np']:
            try:
                var = globals()[var_name]
                if hasattr(var, '__len__') and (isinstance(var, pd.DataFrame) or isinstance(var, list) and len(var) > 1000):
                    del globals()[var_name]
            except:
                pass
    
    # Colectar nuevamente
    gc.collect()
    
    memoria_final = process.memory_info().rss / (1024 ** 3)
    liberado = memoria_inicial - memoria_final
    
    print(f"🧹 Memoria: {memoria_inicial:.2f}GB → {memoria_final:.2f}GB (liberados {liberado:.2f}GB)")

### 2.14. Función para ejecutar combinación optimizada

#### 2.14.1. Función para ejecutar combinación optimizada V1

In [129]:
# 3.1. Ejecutar Combinación y almacenamiento (VERSIÓN OPTIMIZADA)
def ejecutar_combinacion_optimizada():
    """
    Ejecuta el proceso completo de combinación con gestión optimizada de memoria
    """
    
    print("🧹 INICIANDO LIMPIEZA DE MEMORIA...")
    limpiar_memoria_completa()
    
    # ============================================================================
    # PASO 1: CARGAR ARCHIVOS CON GESTIÓN DE MEMORIA
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"📁 PASO 1: CARGANDO ARCHIVOS")
    print(f"{'='*80}")
    
    # Cargar archivos con gestión de memoria
    print("📥 Cargando archivos individuales...")
    archivos_cargados = []
    
    for idx, archivo in enumerate(archivos_a_leer, start=1):
        try:
            ruta_completa = os.path.join("data/processed", archivo)
            if os.path.exists(ruta_completa):
                df_temp = pd.read_parquet(ruta_completa)
                df_temp["ID"] = idx  # Agregar ID
                
                nombre_var = os.path.splitext(archivo)[0]
                globals()[nombre_var] = df_temp
                archivos_cargados.append(archivo)
                
                print(f"   ✅ {nombre_var} cargado ({len(df_temp):,} filas)")
                
                # Liberar memoria temporal
                del df_temp
                gc.collect()
                
            else:
                print(f"   ⚠️  Archivo no encontrado: {archivo}")
                
        except Exception as e:
            print(f"   ❌ Error cargando {archivo}: {e}")
    
    print(f"\n📊 Archivos cargados exitosamente: {len(archivos_cargados)} de {len(archivos_a_leer)}")
    
    # ============================================================================
    # PASO 2: CONVERTIR LISTA A TUPLAS
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🔄 PASO 2: PREPARANDO ESTRUCTURA DE DATOS")
    print(f"{'='*80}")
    
    lista_dataframes = convertir_lista_archivos_a_tuplas(archivos_a_leer)
    
    # ============================================================================
    # PASO 3: CARGAR DICCIONARIO DE HOMOLOGACIÓN
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"📖 PASO 3: CARGANDO DICCIONARIO DE HOMOLOGACIÓN")
    print(f"{'='*80}")
    
    ruta_excel = "data/raw/Referenciales/Fuentes de Información Recolección Inicial.xlsx"
    
    if not os.path.exists(ruta_excel):
        print(f"❌ ERROR: No se encuentra el archivo Excel en {ruta_excel}")
        return None
    
    diccionario, id_estandar = cargar_diccionario_homologacion(
        ruta_excel, 
        "Campos Defunciones"
    )
    
    if diccionario is None:
        print("❌ No se pudo cargar el diccionario de homologación")
        return None
    
    # ============================================================================
    # PASO 4: DIAGNÓSTICO (OPCIONAL)
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🔍 PASO 4: EJECUTANDO DIAGNÓSTICO")
    print(f"{'='*80}")
    
    problemas = diagnosticar_duplicados_columnas(lista_dataframes, diccionario)
    
    if problemas:
        print(f"\n⚠️  Se detectaron {len(problemas)} problemas de duplicados")
        print("💡 El proceso intentará consolidarlos automáticamente")
        
        respuesta = input("\n¿Deseas continuar? (s/n): ")
        if respuesta.lower() != 's':
            print("Proceso cancelado por el usuario")
            return None
    else:
        print("✅ No se detectaron problemas críticos")
    
    # ============================================================================
    # PASO 5: EJECUTAR COMBINACIÓN OPTIMIZADA
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🚀 PASO 5: EJECUTANDO COMBINACIÓN OPTIMIZADA")
    print(f"{'='*80}")
    
    # Limpieza final antes del procesamiento pesado
    print("🧹 Limpieza final de memoria...")
    limpiar_memoria_completa()
    
    try:
        defunciones = homologar_y_combinar_defunciones_optimizado_v2(
            lista_dataframes=lista_dataframes,
            diccionario_homolog=diccionario,
            nombres_estandar=id_estandar,
            ruta_salida="data/processed/defunciones_completo.parquet",
            batch_size=1  # Máxima seguridad para memoria
        )
        
        if defunciones is not None:
            print(f"\n🎉 ¡PROCESO COMPLETADO EXITOSAMENTE!")
            
            # Mostrar estadísticas finales
            print(f"\n{'='*80}")
            print(f"📊 ESTADÍSTICAS FINALES DEL DATASET COMBINADO")
            print(f"{'='*80}")
            print(f"   • Total de registros: {len(defunciones):,}")
            print(f"   • Total de columnas: {len(defunciones.columns)}")
            print(f"   • Memoria utilizada: {defunciones.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
            
            # Información de completitud
            total_celdas = defunciones.shape[0] * defunciones.shape[1]
            celdas_no_nulas = defunciones.notna().sum().sum()
            porcentaje_lleno = (celdas_no_nulas / total_celdas) * 100
            
            print(f"   • Completitud de datos: {porcentaje_lleno:.2f}%")
            print(f"   • Archivo guardado en: data/processed/defunciones_completo.parquet")
            print(f"{'='*80}")
            
            return defunciones
        else:
            print("❌ El proceso de combinación retornó None")
            return None
            
    except Exception as e:
        print(f"\n💥 ERROR CRÍTICO DURANTE LA COMBINACIÓN: {e}")
        print("🔧 Intentando recuperación...")
        
        # Intentar recuperar archivos temporales si existen
        archivos_temp = [f for f in os.listdir("data/processed") if f.startswith("temp_lote_")]
        if archivos_temp:
            print(f"📦 Se encontraron {len(archivos_temp)} archivos temporales")
            print("💡 Puedes intentar combinarlos manualmente")
        
        return None
    
    finally:
        # Limpieza final garantizada
        print("\n🧹 LIMPIEZA FINAL DE MEMORIA...")
        limpiar_memoria_completa()

#### 2.14.2. Función para ejecutar combinación optimizada V2

In [130]:
# 3.1. Ejecutar Combinación y almacenamiento (VERSIÓN CORREGIDA)
def ejecutar_combinacion_optimizada_corregida():
    """
    Ejecuta el proceso completo de combinación con gestión optimizada de memoria
    VERSIÓN CORREGIDA: No elimina DataFrames prematuramente
    """
    
    print("🧹 INICIANDO LIMPIEZA DE MEMORIA...")
    limpiar_memoria_completa()
    
    # ============================================================================
    # PASO 1: CARGAR ARCHIVOS CON GESTIÓN DE MEMORIA
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"📁 PASO 1: CARGANDO ARCHIVOS")
    print(f"{'='*80}")
    
    archivos_a_leer = [
        "defunciones_1979_1991_procesado.parquet",
        "defunciones_1992_1996_procesado.parquet", 
        "defunciones_1997_1997_procesado.parquet",
        "defunciones_1998_2007_procesado.parquet",
        "defunciones_2008_2011_procesado.parquet",
        "defunciones_2012_2013_procesado.parquet",
        "defunciones_2014_procesado.parquet",
        "defunciones_2015_procesado.parquet",
        "defunciones_2016_procesado.parquet",
        "defunciones_2017_procesado.parquet",
        "defunciones_2018_procesado.parquet",
        "defunciones_2019_procesado.parquet",
        "defunciones_2020_procesado.parquet",
        "defunciones_2021_procesado.parquet",
        "defunciones_2022_procesado.parquet",
        "defunciones_2023_procesado.parquet",
        "defunciones_2024_procesado.parquet"
    ]
    
    # Cargar archivos usando la función original que SÍ funciona
    print("📥 Cargando archivos usando función probada...")
    cargar_archivos_seleccionados_con_id("data/processed", archivos_a_leer)
    
    # ============================================================================
    # PASO 2: CONVERTIR LISTA A TUPLAS
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🔄 PASO 2: PREPARANDO ESTRUCTURA DE DATOS")
    print(f"{'='*80}")
    
    lista_dataframes = convertir_lista_archivos_a_tuplas(archivos_a_leer)
    
    # ============================================================================
    # PASO 3: CARGAR DICCIONARIO DE HOMOLOGACIÓN
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"📖 PASO 3: CARGANDO DICCIONARIO DE HOMOLOGACIÓN")
    print(f"{'='*80}")
    
    ruta_excel = "data/raw/Referenciales/Fuentes de Información Recolección Inicial.xlsx"
    
    if not os.path.exists(ruta_excel):
        print(f"❌ ERROR: No se encuentra el archivo Excel en {ruta_excel}")
        return None
    
    diccionario, id_estandar = cargar_diccionario_homologacion(
        ruta_excel, 
        "Campos Defunciones"
    )
    
    if diccionario is None:
        print("❌ No se pudo cargar el diccionario de homologación")
        return None
    
    # ============================================================================
    # PASO 4: DIAGNÓSTICO (OPCIONAL)
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🔍 PASO 4: EJECUTANDO DIAGNÓSTICO")
    print(f"{'='*80}")
    
    problemas = diagnosticar_duplicados_columnas(lista_dataframes, diccionario)
    
    if problemas:
        print(f"\n⚠️  Se detectaron {len(problemas)} problemas de duplicados")
        print("💡 El proceso intentará consolidarlos automáticamente")
        
        respuesta = input("\n¿Deseas continuar? (s/n): ")
        if respuesta.lower() != 's':
            print("Proceso cancelado por el usuario")
            return None
    else:
        print("✅ No se detectaron problemas críticos")
    
    # ============================================================================
    # PASO 5: EJECUTAR COMBINACIÓN CON FUNCIÓN ORIGINAL MEJORADA
    # ============================================================================
    print(f"\n{'='*80}")
    print(f"🚀 PASO 5: EJECUTANDO COMBINACIÓN CON GESTIÓN DE MEMORIA")
    print(f"{'='*80}")
    
    # Limpieza final antes del procesamiento pesado
    print("🧹 Limpieza final de memoria...")
    gc.collect()
    
    try:
        # Usar la función original pero con batch_size más conservador
        defunciones = homologar_y_combinar_defunciones_optimizado(
            lista_dataframes=lista_dataframes,
            diccionario_homolog=diccionario,
            nombres_estandar=id_estandar,
            ruta_salida="data/processed/defunciones_completo.parquet",
            batch_size=1  # Procesar uno a la vez
        )
        
        if defunciones is not None:
            print(f"\n🎉 ¡PROCESO COMPLETADO EXITOSAMENTE!")
            
            # Mostrar estadísticas finales
            print(f"\n{'='*80}")
            print(f"📊 ESTADÍSTICAS FINALES DEL DATASET COMBINADO")
            print(f"{'='*80}")
            print(f"   • Total de registros: {len(defunciones):,}")
            print(f"   • Total de columnas: {len(defunciones.columns)}")
            print(f"   • Memoria utilizada: {defunciones.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
            
            # Información de completitud
            total_celdas = defunciones.shape[0] * defunciones.shape[1]
            celdas_no_nulas = defunciones.notna().sum().sum()
            porcentaje_lleno = (celdas_no_nulas / total_celdas) * 100
            
            print(f"   • Completitud de datos: {porcentaje_lleno:.2f}%")
            print(f"   • Archivo guardado en: data/processed/defunciones_completo.parquet")
            
            # Mostrar primeras filas
            print(f"\n📋 MUESTRA DEL DATASET (primeras 2 filas):")
            print(defunciones.head(2))
            print(f"{'='*80}")
            
            return defunciones
        else:
            print("❌ El proceso de combinación retornó None")
            print("💡 Intentando con enfoque alternativo...")
            return ejecutar_combinacion_alternativa(lista_dataframes, diccionario, id_estandar)
            
    except Exception as e:
        print(f"\n💥 ERROR CRÍTICO DURANTE LA COMBINACIÓN: {e}")
        print("🔧 Intentando enfoque alternativo...")
        
        # Intentar con enfoque simplificado
        try:
            return ejecutar_combinacion_alternativa(lista_dataframes, diccionario, id_estandar)
        except Exception as e2:
            print(f"💥 Error también en enfoque alternativo: {e2}")
            return None
    
    finally:
        # Limpieza final garantizada
        print("\n🧹 LIMPIEZA FINAL DE MEMORIA...")
        limpiar_memoria_completa()

### 2.15. Función para ejecutar recuperación para casos de fallo

In [131]:
# ============================================================================
# FUNCIÓN DE RECUPERACIÓN PARA CASOS DE FALLO
# ============================================================================

def recuperar_proceso_fallido():
    """
    Función para recuperar el proceso si falla la combinación principal
    """
    print(f"\n{'='*80}")
    print(f"🔄 MODO DE RECUPERACIÓN")
    print(f"{'='*80}")
    
    # Buscar archivos temporales
    archivos_temp = []
    for i in range(1, 100):  # Buscar hasta 100 lotes
        temp_file = f"data/processed/temp_lote_{i:03d}.parquet"
        if os.path.exists(temp_file):
            archivos_temp.append(temp_file)
    
    if not archivos_temp:
        print("❌ No se encontraron archivos temporales para recuperar")
        return None
    
    print(f"📦 Encontrados {len(archivos_temp)} archivos temporales")
    
    # Combinar archivos temporales
    chunks = []
    for archivo in archivos_temp:
        print(f"   📥 Cargando: {archivo}")
        df_temp = pd.read_parquet(archivo)
        chunks.append(df_temp)
    
    print("🔗 Combinando archivos temporales...")
    df_final = pd.concat(chunks, ignore_index=True)
    
    # Guardar resultado final
    ruta_final = "data/processed/defunciones_completo_recuperado.parquet"
    guardar_dataframe_seguro(df_final, ruta_final)
    
    # Limpiar temporales
    for archivo in archivos_temp:
        try:
            os.remove(archivo)
            print(f"   🧹 Eliminado: {archivo}")
        except:
            print(f"   ⚠️  No se pudo eliminar: {archivo}")
    
    print(f"✅ Proceso de recuperación completado")
    print(f"💾 Archivo guardado como: {ruta_final}")
    
    return df_final

### 2.16. Función principal con manejo de errores

#### 2.16.1. Función principal V1

In [132]:
# ============================================================================
# EJECUCIÓN PRINCIPAL CON MANEJO DE ERRORES
# ============================================================================

def main():
    """
    Función principal con manejo robusto de errores
    """
    lista_dataframes = []
    try:
        print(f"{'='*80}")
        print(f"🎯 INICIANDO PROCESO DE COMBINACIÓN DE DEFUNCIONES")
        print(f"{'='*80}")
        
        # Verificar que existe la carpeta de processed
        os.makedirs("data/processed", exist_ok=True)
        
        # Ejecutar proceso principal
        resultado = ejecutar_combinacion_optimizada()
        
        if resultado is not None:
            print(f"\n✅ ¡PROCESO TERMINADO CON ÉXITO!")
            print(f"📊 Dataset final creado con {len(resultado):,} registros")
            
            # Mostrar muestra del resultado
            print(f"\n📋 MUESTRA DEL DATASET FINAL (primeras 3 filas):")
            print(resultado.head(3))
            
            return resultado
        else:
            print(f"\n⚠️  El proceso no produjo resultados")
            
            # Ofrecer recuperación
            respuesta = input("¿Intentar recuperación de archivos temporales? (s/n): ")
            if respuesta.lower() == 's':
                resultado_recuperado = recuperar_proceso_fallido()
                return resultado_recuperado
            else:
                return None
                
    except Exception as e:
        print(f"\n💥 ERROR NO CONTROLADO: {e}")
        print("🔄 Intentando recuperación automática...")
        
        resultado_recuperado = recuperar_proceso_fallido()
        return resultado_recuperado
    
    finally:
        print(f"\n{'='*80}")
        print(f"🏁 PROCESO FINALIZADO")
        print(f"{'='*80}")

#### 2.16.2. Función principal V2

In [133]:
# ============================================================================
# EJECUCIÓN PRINCIPAL CORREGIDA
# ============================================================================

def main_corregido():
    """
    Función principal corregida
    """

    try:
        print(f"{'='*80}")
        print(f"🎯 INICIANDO PROCESO DE COMBINACIÓN (VERSIÓN CORREGIDA)")
        print(f"{'='*80}")

        # =============================================================
        # 🔹 Cargar diccionario de homologación
        # =============================================================
        print("\n📘 Cargando diccionario de homologación...")
        diccionario, id_estandar = cargar_diccionario_homologacion(
            "data/raw/Referenciales/Fuentes de Información Recolección Inicial.xlsx",
            "Campos Defunciones"
            )
        print(f"✅ Diccionario cargado correctamente con ID estándar: {id_estandar}")
        
        # Verificar que existe la carpeta de processed
        os.makedirs("data/processed", exist_ok=True)
        
        # Ejecutar proceso principal CORREGIDO
        resultado = ejecutar_combinacion_optimizada_corregida()
        
        if resultado is not None:
            print(f"\n✅ ¡PROCESO TERMINADO CON ÉXITO!")
            print(f"📊 Dataset final creado con {len(resultado):,} registros y {len(resultado.columns)} columnas")
            
            # Verificar que el archivo se guardó
            if os.path.exists("data/processed/defunciones_completo.parquet"):
                tamaño_mb = os.path.getsize("data/processed/defunciones_completo.parquet") / (1024 * 1024)
                print(f"💾 Archivo guardado: data/processed/defunciones_completo.parquet ({tamaño_mb:.2f} MB)")
            elif os.path.exists("data/processed/defunciones_completo_alternativo.parquet"):
                tamaño_mb = os.path.getsize("data/processed/defunciones_completo_alternativo.parquet") / (1024 * 1024)
                print(f"💾 Archivo guardado: data/processed/defunciones_completo_alternativo.parquet ({tamaño_mb:.2f} MB)")
            
            return resultado
        else:
            print(f"\n❌ El proceso no pudo completarse")
            
            # Último intento con método simplificado
            print("🔄 Intentando método ultra-simplificado...")
            
            # Recargar lista de dataframes usando lista global
            print("🔁 Usando lista global de archivos predefinidos...")
            lista_dataframes = convertir_lista_archivos_a_tuplas(archivos_a_leer)

            # Cargar los archivos (solo para verificación)
            cargar_archivos_seleccionados_con_id("data/processed", archivos_a_leer)

            print(f"\n📋 Verificando lista de dataframes a combinar...")
            print(lista_dataframes)
            if not lista_dataframes:
                raise ValueError("❌ La lista de dataframes está vacía. No hay archivos para combinar.")
            
            resultado_final = combinacion_simplificada_memoria(
                lista_dataframes, 
                diccionario, 
                id_estandar, 
                "data/processed/defunciones_completo_final.parquet"
            )
            
            return resultado_final
                
    except Exception as e:
        print(f"\n💥 ERROR NO CONTROLADO: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    finally:
        print(f"\n{'='*80}")
        print(f"🏁 PROCESO FINALIZADO")
        print(f"{'='*80}")


### 2.17. Función combinar parquets en stream

In [134]:
def combinar_parquets_en_stream(lista_archivos, ruta_salida):
    """
    Combina múltiples archivos Parquet grandes sin cargar todo en memoria.
    """
    writer = None
    for i, archivo in enumerate(lista_archivos, start=1):
        print(f"🔄 [{i}/{len(lista_archivos)}] Agregando {archivo}...")
        table = pq.read_table(archivo)

        if writer is None:
            writer = pq.ParquetWriter(ruta_salida, table.schema, compression="snappy")

        writer.write_table(table)

    if writer:
        writer.close()
        print(f"✅ Archivo combinado guardado en {ruta_salida}")

### 2.18. Función para limpiar tipos

In [135]:
# Reemplaza la Celda 25 por esto
def limpiar_tipos(df, columnas_a_string=None):
    """
    Estandariza tipos para evitar errores con pyarrow/fastparquet.
    - columnas_a_string: lista opcional de columnas que FORZAR a string.
    - Convierte columnas numéricas con strings a numeric si es posible.
    - Evita convertir sin necesidad.
    """
    import pandas as pd
    df = df.copy()
    # si se pasó lista explícita, forzar solo esas a string
    if columnas_a_string:
        for col in columnas_a_string:
            if col in df.columns:
                df[col] = df[col].astype(str).replace({"nan": None, "NaN": None, "None": None})
        return df

    # intentar convertir columnas object -> numeric cuando la mayoría parezcan números
    for col in df.columns:
        if df[col].dtype == "object":
            sample = df[col].dropna().astype(str).head(200)
            n_numeric = sample.str.match(r'^-?\d+(\.\d+)?$').sum()
            if len(sample) > 0 and (n_numeric / len(sample) > 0.6):
                # parece numérica -> convertir
                df[col] = pd.to_numeric(df[col], errors='coerce')
            else:
                # si mezcla tipos y hay floats en objeto, convertir a string seguro
                # pero solo si hay mezcla (evitar convertir identifiadores)
                types_present = set(type(x) for x in df[col].dropna().head(200))
                if any(t in (int,float) for t in types_present):
                    df[col] = df[col].astype(str).replace({"nan": None, "NaN": None, "None": None})
    # Para floats que sean realmente enteros sin NaN, convertir a Int64 (nullable) para parquet
    for col in df.select_dtypes(include=['float']).columns:
        # si no hay decimales significativos, intentar Int64
        ser = df[col]
        if ser.dropna().apply(float.is_integer).all():
            df[col] = ser.astype("Int64")
    return df


## 3. PROCESAMIENTO FUNCIONES

### 3.1. Ejecutar Combinación y almacenamiento

In [136]:
# ============================================================================
# EJECUTAR EL PROCESO CORREGIDO
# ============================================================================

print(f"{'='*80}")
print(f"🚀 EJECUTANDO PROCESO CORREGIDO")
print(f"{'='*80}")

# Lista global de archivos parquet a combinar
archivos_a_leer = [
    "defunciones_1979_1991_procesado.parquet",
    "defunciones_1992_1996_procesado.parquet",
    "defunciones_1997_1997_procesado.parquet",
    "defunciones_1998_2007_procesado.parquet",
    "defunciones_2008_2011_procesado.parquet",
    "defunciones_2012_2013_procesado.parquet",
    "defunciones_2014_procesado.parquet",
    "defunciones_2015_procesado.parquet",
    "defunciones_2016_procesado.parquet",
    "defunciones_2017_procesado.parquet",
    "defunciones_2018_procesado.parquet",
    "defunciones_2019_procesado.parquet",
    "defunciones_2020_procesado.parquet",
    "defunciones_2021_procesado.parquet",
    "defunciones_2022_procesado.parquet",
    "defunciones_2023_procesado.parquet",
    "defunciones_2024_procesado.parquet"
]

# Ejecutar proceso principal corregido
dataset_final = main_corregido()

if dataset_final is not None:
    print(f"\n🎊 ¡PROCESO COMPLETADO CON ÉXITO!")
    print(f"💾 Dataset disponible como 'dataset_final'")
    print(f"📊 Dimensión: {dataset_final.shape}")
else:
    print(f"\n😞 El proceso no pudo completarse")
    print("💡 Revisa los mensajes de error específicos")

🚀 EJECUTANDO PROCESO CORREGIDO
🎯 INICIANDO PROCESO DE COMBINACIÓN (VERSIÓN CORREGIDA)

📘 Cargando diccionario de homologación...

🔍 DEBUG - Información del Excel:
   Dimensiones originales: (1048575, 79)
   Columnas totales: 79

🔍 Validando filas del Excel...
   Filas con ID válido: 1048575
   ├─ Con datos en campos: 16
   └─ Sin datos (descartadas): 1048559
      IDs descartados (primeros 10): [17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
      ... y 1048549 más

   ✅ Filas válidas a procesar: 16
   📋 Rango de IDs: 1 - 16
   📋 Campos a procesar: 78

🔍 Detectando nombres estándar (último valor disponible por columna)...
   ✓ 'CÓDIGO DEPARTAMENTO' → 'COD_DPTO'
   ✓ 'CÓDIGO MUNICIPIO' → 'COD_MUNIC'
   ✓ 'ÁREA DEFUNCIÓN' → 'A_DEFUN'
   ✓ 'CÓDIGO IPS' → 'COD_INST'
   ✓ 'NOMBRE IPS' → 'NOM_INST'
   ... y 73 campos más

   📊 Total nombres estándar detectados: 78

🔄 Creando mapeos de homologación por ID...

   ✅ IDs procesados: 16
   ├─ Con mapeos (campos a renombrar): 13
   └─ Sin mapeos (nombres

Traceback (most recent call last):
  File "C:\Users\USUARIO\AppData\Local\Temp\ipykernel_3216\3845605723.py", line 62, in main_corregido
    resultado_final = combinacion_simplificada_memoria(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\USUARIO\AppData\Local\Temp\ipykernel_3216\3585990353.py", line 63, in combinacion_simplificada_memoria
    raise ValueError("❌ No se pudo concatenar: todos los dataframes estaban vacíos o None.")
ValueError: ❌ No se pudo concatenar: todos los dataframes estaban vacíos o None.
